# Process Multiple Long Strips

In [ ]:
#%% imports
from pathlib import Path
# import json
# import configparser
import pprint
pp = pprint.PrettyPrinter(indent=4);
import time
import math
from collections import namedtuple

# # pip install slicerio
# import slicerio.server

# # packages for 3d
# #   # for OCT 3D stuff and using in jupyter
# #   - trame-jupyter-extension
# #   - trame
# #   - trame-vtk
# #   - trame-vuetify
# #   - ipywidgets

import matplotlib.pyplot as plt
import plotly.express as px

# interactive panels
import panel as pn
pn.extension('plotly');

import numpy as np
import pandas as pd
import addict

import cv2
import skimage
import scipy.signal

import pyvista as pv
from PIL import Image
import SimpleITK as sitk

# with vedo
#from vedo import dataurl, Volume, Text2D
import vedo
vedo.settings.default_backend = 'vtk'
#from vedo.applications import Slicer3DPlotter
import naatos_oct_tools.plotters.vedo_plotters as vedo_plotters
import naatos_oct_tools.plotters.sitk_plotters as sitk_plotters

import naatos_oct_tools.thorlabs_oct_file_reading
import naatos_oct_tools.oct_linear_scan_processing

In [ ]:
# Magics to autoreload submodules when they are modified
%load_ext autoreload
%autoreload 2

In [ ]:
#%% Test Record Excel
dftests = pd.read_excel(
    r'C:\Users\SimonGhionea\Global Health Labs, Inc\NAATOS Product Feasibility - TB V1 - General - Internal - Wax Valve\OCT\Test\OCT_wax_valve_test_record.xlsx',
    skiprows=1
)
dftests = dftests.iloc[:,1:]
dftests.rename(columns={'Unnamed: 19':'Notes'},inplace=True)
dftests

In [ ]:
#%% Filter The Tests To Process
dfmasks = [
    (dftests['Test date']<'2025-03-28') & (dftests['Test date']>='2025-03-26'),
    dftests['Test name'] != 'GHL_pyapp_20250326T1258'
]
# dfmasks = [
#     (dftests['Test date']<='2025-04-15') & (dftests['Test date']>='2025-04-14')
# ]

dffilt = dftests[np.all(dfmasks,axis=0)]
dffilt

In [ ]:
#%% Load OCT study information
octstudies = [];
#folder_octexport_root = Path(r'\\file.corp.ghlabs.org\Shared\Projects\NAATOS\V1\NAATOS_OCT_WORK\OCTExport')
folder_octexport_root = Path(r'D:\SGProjects\NAATOS\OCTlocal')

# Load list of data
for idx,record in dffilt.iterrows():
    octstudy = naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder(record['Test name'],folder_octexport_root);
    octstudies.append(octstudy);


In [ ]:
# Master Parameters
oct_scalar_min = 30;
oct_scalar_max = 60;

In [ ]:
#%% Load OCTSTUDY object, and start some processing on it
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print(octstudy)

    fname_merged_and_rescaled_volume = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
    
    if(octstudy.study_info['study_num_oct_files']>0):
        # --Loading And Pre-Processing--
        if(fname_merged_and_rescaled_volume.exists()):
            print(f'Loading {fname_merged_and_rescaled_volume.name}')
            # Load the strip and merge into one volume
            vdvol = vedo.Volume(pv.read(fname_merged_and_rescaled_volume));
            octstudy.vdvol = vdvol;
        else:
            # Load OCT Data for this study
            octstudy.load_all_octs();


            # THESE WILL DO NOTHING IF ANTICIPATED OUTPUTS/ARTIFACTS ALREADY EXIST IN THE PROCESSED FOLDER

            # RGB Camera Images - Write them out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_individual_images(octstudy);

            # RGB Camera Images - Make a montage and write out as .jpg to processed folder
            naatos_oct_tools.oct_linear_scan_processing.process_rgbcamera_and_make_montage_image(octstudy);
            #break;


            # IF MERGED STACKED AND RESCALED TO SCALAR RANGE FILE EXISTS, LOAD THAT; OTHERWISE PROCESS IT HERE
            print(f'Generating merged and rescaled .vtk volume');
            # Generate merged and rescaled volume
            vdvol = octstudy.generate_merged_vdvol_and_rescaled(oct_scalar_min,oct_scalar_max);
            octstudy.vdvol = vdvol;

            # save this byte-adjusted volume
            if True:
                octstudy.folder_study_processed.mkdir(exist_ok=True);
                fname = octstudy.folder_study_processed/'{:s}_STACKED_RESCALED_{:}to{:}_uint8.vtk'.format(octstudy.name,oct_scalar_min,oct_scalar_max);
                #vdvol.dataset.save()
                if(fname.exists()):
                    print('Stacked volume byte-size .vtk file already exists, will not recreate.');
                    print(fname);
                else:
                    vdvol.dataset.save(fname);
            
            # Unload OCT Data for this study (we will still keep the vdvol)
            octstudy.unload_all_octdata();

        # --Detailed Image Processing--
        del octstudy.vdvol;
        #break;
    

In [ ]:
#a = octstudy.load_all_octs()

# Call The Notebooks

In [17]:
#%% Load OCTSTUDY object, and start some processing on it
octstudies_to_run = [];
for idx,octstudy in enumerate(octstudies):
    octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder;
    print('~~~~~~~');
    print(octstudy.name);
    pp.pprint(octstudy.resultsCheck())
    doWeRunTheNotebook = any([v is False for k,v in octstudy.resultsCheck().items()])
    print('Run?',doWeRunTheNotebook)
    if(doWeRunTheNotebook):
        octstudies_to_run.append(octstudy);

print('~~~~~~~');
print('We will run the processing on {:} octstudies.'.format(len(octstudies_to_run)))
print([x.name for x in octstudies_to_run])

~~~~~~~
GHL_pyapp_20250326T1403
{   'along_strip_data_extracted': [   '/dfstepA',
                                      '/dfstepB',
                                      '/dfstepC',
                                      '/dfstepD'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True,
    'figoutmp4_stepB': True}
Run? False
~~~~~~~
GHL_pyapp_20250326T1416
{   'along_strip_data_extracted': ['/dfstepA', '/dfstepB', '/dfstepC'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True,
    'figoutmp4_stepB': True}
Run? False
~~~~~~~
GHL_pyapp_20250327T1428
{   'along_strip_data_extracted': [   '/dfstepA',
                                      '/dfstepB',
                                      '/dfstepC',
                                      '/dfstepD'],
    'exist_data_extracted': True,
    'figoutmp4_stepA': True,
    'figoutmp4_stepB': True}
Run? False
~~~~~~~
GHL_pyapp_20250327T1438
{   'along_strip_data_extracted': [   '/dfstepA',
                                      '

In [18]:
import papermill

#nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\nb_test_nbparams.ipynb";
nbpath = r"C:\Users\SimonGhionea\OneDrive - Global Health Labs, Inc\ProjectsCloud\NAATOS\sgpyanalysisoct\sandbox_sg\octproc_20250429_process_a_longstrip.ipynb";

parameters = papermill.inspect_notebook(nbpath)
pp.pprint(parameters);

{   'codename': {   'default': "'20250429';",
                    'help': '',
                    'inferred_type_name': 'None',
                    'name': 'codename'},
    'oct_scalar_max': {   'default': '60;',
                          'help': '',
                          'inferred_type_name': 'None',
                          'name': 'oct_scalar_max'},
    'oct_scalar_min': {   'default': '30;',
                          'help': '',
                          'inferred_type_name': 'None',
                          'name': 'oct_scalar_min'},
    'study_name': {   'default': "'GHL_pyapp_20250327T1428';",
                      'help': 'oct study folder',
                      'inferred_type_name': 'None',
                      'name': 'study_name'}}


# Call notebooks to process (serially)

In [ ]:
for octstudy in octstudies_to_run:
    print(f'Now launching the processing notebook on {octstudy.name}');
    
    # set output path
    #nbpath_out = Path(nbpath).parent/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    nbpath_out = Path(r'C:\TEMP\OCTtmp')/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    print(nbpath_out);

    # execute a notebook
    papermill.execute_notebook(
        input_path=nbpath,
        output_path=nbpath_out,
        parameters=dict(codename = Path(nbpath).name,study_name=octstudy.name)
    )

# Call notebooks to process (Multiple-strips in parallel, concurrent futures)

In [ ]:
import concurrent.futures

def run_notebook_on_an_octstudy(octstudy : naatos_oct_tools.thorlabs_oct_file_reading.OCT_Study_Folder):
    # set output path
    #nbpath_out = Path(nbpath).parent/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    nbpath_out = Path(r'C:\TEMP\OCTtmp')/(Path(nbpath).stem+'_OUT{:s}.ipynb').format(octstudy.name)
    print(nbpath_out);

    # # execute a notebook
    papermill.execute_notebook(
        input_path=nbpath,
        output_path=nbpath_out,
        parameters=dict(codename = Path(nbpath).name,study_name=octstudy.name)
    )
    # print('start',octstudy.name);
    # time.sleep(1)
    # print('stop',octstudy.name)

# Using concurrent.futures to handle multiprocessing
def run_in_parallel(num_processes):
    # with concurrent.futures.ProcessPoolExecutor(max_workers=num_processes) as executor:
    #     futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
    #     for future in concurrent.futures.as_completed(futures):
    #         print(future.result())  # You can handle results or exceptions here
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(run_notebook_on_an_octstudy, octstudy) for octstudy in octstudies_to_run]
        for future in concurrent.futures.as_completed(futures):
            print(future.result())

# Example usage
num_processes = 2  # Number of parallel processes
run_in_parallel(num_processes)

In [ ]:
import concurrent.futures
import time

def task(n):
    time.sleep(1)
    return n * n

# Using ThreadPoolExecutor
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(task, i) for i in range(10)]
    for future in concurrent.futures.as_completed(futures):
        print(future.result())

# # Using ProcessPoolExecutor
# with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
#     results = executor.map(task, range(10))
#     for result in results:
#         print(result)

# Processing Step 0 - Replace VTK scalars with scaled bytes

In [ ]:
#del vdvol;
#del scalars_rescaled_as_int

# Develop VTK Range Rescaling and integer

In [ ]:
def doVtkImageMinMaxScaling(vtkimagedata):
    # restrict range and re-scale, cast to integer (will reduce memory by 4x)
    oct_scalar_min = 30;
    oct_scalar_max = 60;

    import vtk

    # Create VTK Pipeline Connections
    # 1. Thresholding Hi
    alg_thresholder = vtk.vtkImageThreshold();
    alg_thresholder.SetInputData(vtkimagedata)
    alg_thresholder.ThresholdByUpper(oct_scalar_min);
    alg_thresholder.SetOutValue(oct_scalar_min);
    alg_thresholder.ReplaceInOff();
    alg_thresholder.ReplaceOutOn();

    # 2. Thresholding Lo
    alg_thresholder2 = vtk.vtkImageThreshold();
    alg_thresholder2.SetInputConnection(alg_thresholder.GetOutputPort());
    alg_thresholder2.ThresholdByLower(oct_scalar_max);
    alg_thresholder2.SetOutValue(oct_scalar_max);
    alg_thresholder2.ReplaceInOff();
    alg_thresholder2.ReplaceOutOn();

    # 3. Subtract Minimum Value
    alg_math1 = vtk.vtkImageMathematics();
    alg_math1.SetInputConnection(alg_thresholder2.GetOutputPort());
    alg_math1.SetConstantC(-oct_scalar_min);
    alg_math1.SetOperationToAddConstant();

    # 4. Rescale to 0 to 255
    alg_math2 = vtk.vtkImageMathematics();
    alg_math2.SetInputConnection(alg_math1.GetOutputPort());
    alg_math2.SetConstantK(255.0/oct_scalar_min);
    alg_math2.SetOperationToMultiplyByK();
    #cast to unit8_t byte
    #alg_math2.SetOutputS();

    # 5. Cast to Uint8
    alg_caster = vtk.vtkImageCast();
    alg_caster.SetInputConnection(alg_math2.GetOutputPort());
    alg_caster.SetOutputScalarTypeToUnsignedChar();


    # execute the vtk pipline, wrap with pyvista, and return
    alg_final = alg_caster;
    alg_final.Update();

    final = pv.wrap(alg_final.GetOutput())
    return final;
mergedvol_rescaled = doVtkImageMinMaxScaling(mergedvol);

In [ ]:
np.max(mergedvol.active_scalars)

In [ ]:
print(np.min(mergedvol_rescaled.active_scalars),np.max(mergedvol_rescaled.active_scalars))

In [ ]:
mergedvol_rescaled['OCTintensity']

In [ ]:
del mergedvol,mergedvol_rescaled,alg_thresholder

In [ ]:
del alg_final